# 🔐 LangChain Tools — Context (runtime.context)
### Passing Immutable Configuration Data to Tools at Invocation Time
> **Python 3.11 | LangChain v1.0.0 | .env config**


## Cell 1 — Install & Imports

In [ ]:
%pip install langchain-openai python-dotenv langgraph -q

from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
import os
from dotenv import load_dotenv

load_dotenv()
print("✅ Ready!")

## Cell 2 — LLM Setup

In [ ]:
llm = ChatOpenAI(
    model=os.getenv("MODEL"),
    base_url=os.getenv("API_URL"),
    api_key=os.getenv("API_KEY"),
    temperature=0,
)
print("✅ LLM configured!")

## Cell 3 — What is Context?

Context = **immutable** configuration data passed at invocation time.

Use it for things that:
- ✅ Should **NOT** change during a conversation
- ✅ Should **NOT** be visible to the LLM
- ✅ Identify **WHO** is calling — user ID, session ID, tenant ID

| Feature      | State                        | Context                        |
|--------------|------------------------------|--------------------------------|
| Mutability   | Mutable (can change)         | Immutable (fixed per invoke)   |
| Access       | `runtime.state["key"]`       | `runtime.context.field_name`   |
| Update       | `Command(update={...})`      | ❌ Cannot be updated           |
| Passed at    | `agent.invoke({"messages"})` | `agent.invoke(..., context=)`  |
| LLM sees it? | ❌ No                        | ❌ No                          |
| Use for      | History, counters, flags     | User ID, session ID, tenant ID |


## Cell 4 — Define Dummy Database

In [ ]:
USER_DATABASE = {
    "user123": {
        "name"        : "Alice Johnson",
        "account_type": "Premium",
        "balance"     : 5000,
        "email"       : "alice@example.com",
    },
    "user456": {
        "name"        : "Bob Smith",
        "account_type": "Standard",
        "balance"     : 1200,
        "email"       : "bob@example.com",
    },
    "user789": {
        "name"        : "Priya Sharma",
        "account_type": "Premium",
        "balance"     : 8750,
        "email"       : "priya@example.com",
    },
}

print("✅ Dummy database ready!")
print(f"   Users: {list(USER_DATABASE.keys())}") 

## Cell 5 — Define Context Schema

In [ ]:
# Context is a simple dataclass
# Fields are passed at invoke() time and stay fixed throughout

@dataclass
class UserContext:
    user_id    : str
    session_id : str = "default-session"   # optional with default

print("✅ UserContext schema defined!")
print(f"   Fields: user_id, session_id")

## Cell 6 — Define Tools that ACCESS Context

In [ ]:
# runtime.context gives access to UserContext fields
# 'runtime' is completely hidden from the LLM

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id          # ← read from context

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return (
            f"Account Holder : {user['name']}\n"
            f"Account Type   : {user['account_type']}\n"
            f"Balance        : ${user['balance']}"
        )
    return f"User '{user_id}' not found."


@tool
def get_email(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's registered email address."""
    user_id = runtime.context.user_id          # ← read from context

    if user_id in USER_DATABASE:
        return f"Registered email: {USER_DATABASE[user_id]['email']}"
    return f"User '{user_id}' not found."


@tool
def get_session_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current session information."""
    return (
        f"User ID    : {runtime.context.user_id}\n"
        f"Session ID : {runtime.context.session_id}"
    )


print("✅ Tools defined!")
print(f"\n📋 Args visible to LLM:")
for t in [get_account_info, get_email, get_session_info]:
    props = list(t.args_schema.schema().get("properties", {}).keys())
    print(f"   {t.name:<22} → {props if props else '[] (runtime is hidden)'}") 

## Cell 7 — Create Agent with Context Schema

In [ ]:
tools = [get_account_info, get_email, get_session_info]

agent = create_agent(
    llm,
    tools=tools,
    context_schema=UserContext,           # ← register context schema here
    system_prompt=(
        "You are a financial assistant. "
        "Use tools to fetch the user's account details. "
        "Never ask the user for their user ID — it is already known."
    )
)

print("✅ Agent created with context schema!")
print(f"   Tools: {[t.name for t in tools]}") 

## Cell 8 — Invoke as User 1 (Alice)

In [ ]:
# Context is passed at invoke() time
# This sets WHO is calling the agent

result = agent.invoke(
    {"messages": [HumanMessage(content="What's my current balance?")]},
    context=UserContext(user_id="user123", session_id="sess-abc")
)

print("👤 Invoking as user123 (Alice)")
print("-" * 45)
print("🤖", result["messages"][-1].content)

## Cell 9 — Invoke as User 2 (Bob)

In [ ]:
# Same agent, same tools — different context → different user's data

result = agent.invoke(
    {"messages": [HumanMessage(content="What's my current balance?")]},
    context=UserContext(user_id="user456", session_id="sess-xyz")
)

print("👤 Invoking as user456 (Bob)")
print("-" * 45)
print("🤖", result["messages"][-1].content)

## Cell 10 — Invoke: Get Email

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="What email address do I have on file?")]},
    context=UserContext(user_id="user789", session_id="sess-pqr")
)

print("👤 Invoking as user789 (Priya)")
print("-" * 45)
print("🤖", result["messages"][-1].content)

## Cell 11 — Invoke: Get Session Info

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="Can you show me my session details?")]},
    context=UserContext(user_id="user123", session_id="sess-demo-001")
)

print("👤 Invoking as user123 (Alice)")
print("-" * 45)
print("🤖", result["messages"][-1].content)

## ✅ Summary

### Define Context
```python
@dataclass
class MyContext:
    user_id    : str
    session_id : str = "default"
```

### Access in Tool
```python
@tool
def my_tool(runtime: ToolRuntime[MyContext]) -> str:
    uid = runtime.context.user_id   # ← immutable, hidden from LLM
```

### Register & Invoke
```python
agent = create_agent(model, tools, context_schema=MyContext)

agent.invoke(
    {"messages": [HumanMessage(content="...")]},
    context=MyContext(user_id="u123", session_id="s001")
)
```

### 🔑 Key Rules
- Context is **immutable** — use `Command` if you need to update things
- Context is **hidden** from the LLM — LLM never sees user_id
- Context is **per invocation** — each `.invoke()` can have a different context
- Perfect for **multi-user apps** where each user gets their own data
